In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity


C:\Users\lenovo\AppData\Local\Temp\ipykernel_15332\901470665.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
e:\RAG\TRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: FYP_Proposal_AI_University_Management_System.pdf
  ✓ Loaded 13 pages

Total documents loaded: 13


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(all_pdf_documents)

print(f"Total pages/documents: {len(all_pdf_documents)}")
print(f"Total chunks created: {len(chunks)}")


Total pages/documents: 13
Total chunks created: 33


In [4]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2248.71it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\lenovo\AppData\Local\Temp\ipykernel_15332\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [5]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [6]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-02T10:30:37+00:00', 'title': 'Final Year Report Template', 'author': 'Barry Smyth', 'moddate': '2026-09-02T10:30:37+00:00', 'source': '..\\data\\pdf\\FYP_Proposal_AI_University_Management_System.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'FYP_Proposal_AI_University_Management_System.pdf', 'file_type': 'pdf'}, page_content='AI Powered University Management System \nFinal Year Project Proposal \nSession 2023-2027 \n \n \nA project submitted in partial fulfilment of the requirements for the \nDegree \nof  \nBS in Computer Science / Software Engineering / Artificial Intelligence \n \n \n \nDepartment of Computer Science \nCOMSATS University Islamabad (CUI), Lahore Campus'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-02T10:30:37+00:00', 'title': 'Final Year Report Template', 'author': '

In [7]:
#convert the text to embeddings

texts = [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)


## store in the vector database
vectorstore.add_documents(chunks , embeddings)

Generating embeddings for 33 texts...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.76s/it]


Generated embeddings with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 33


### Retriever Pipeline From VectorStore

In [8]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 10,
        max_distance: float = None
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: Search query
            top_k: Number of results to retrieve
            max_distance: Optional maximum allowed distance.
                          Lower distance = more similar.

        Returns:
            List of dictionaries containing documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Max distance: {max_distance}")

        # 1. Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:
            # 2. Search ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            # 3. Check if results exist
            if not results["documents"] or not results["documents"][0]:
                print("No documents found")
                return []

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            # 4. Process results
            for rank, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances),
                start=1
            ):

                # Lower distance means better match
                if max_distance is not None and distance > max_distance:
                    continue

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "distance": distance,
                    "rank": rank
                })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                f"(after filtering)"
            )

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Create retriever
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)


In [9]:
rag_retriever

In [10]:
rag_retriever.retrieve("What is the abstract of project")

Retrieving documents for query: 'What is the abstract of project'
Top K: 10, Max distance: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.37it/s]

Generated embeddings with shape: (1, 384)
Retrieved 10 documents (after filtering)


[{'id': 'doc_c3f09532_2',
  'content': 'Project ID (for office use)   \nType of project [  ] Traditional              [  ] Industrial  [✓ ] Continuing \nNature of project [✓ ] Development            [  ] Research & Development  \nSustainable Development \nGoals(SDGs) \n[  ] Good Health and Well-Being                      [✓ ] Quality Education  \n[✓ ]  Industry, Innovation, and Infrastructure     [  ] Gender Equality \n[  ] Decent Work and Economic Growth           [  ]  Climate Action \nArea of specialization \n[✓ ] Artificial Intelligence (AI)      [  ] Blockchain               [  ] Cybersecurity \n[  ] Data Science and Analytics    [  ] Game Development \n[  ] Internet of Things (IoT)          [✓ ]  Natural Language Processing (NLP) \n[  ] Mobile App Development      [✓ ]  Web Development \nProject Group Members \nSr.# Reg. # Student Name Email ID Phone # Signature \n(i) \nGroup Leader \nSP23-BSE-135 \nMuhammad Zaid Amjad sp23-bse-135@cuilahore.edu.pk 03100044108  \n(ii) SP23-BSE-00

In [11]:
print(vectorstore.collection.count())

33


In [12]:
query = "What is the abstract of project"

query_embedding = embedding_manager.generate_embeddings([query])[0]

results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print(results)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.57it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_c3f09532_2', 'doc_213858b5_22', 'doc_a7bb9f6b_0', 'doc_530b5a97_5', 'doc_c804da24_26']], 'embeddings': None, 'documents': [['Project ID (for office use)   \nType of project [  ] Traditional              [  ] Industrial  [✓ ] Continuing \nNature of project [✓ ] Development            [  ] Research & Development  \nSustainable Development \nGoals(SDGs) \n[  ] Good Health and Well-Being                      [✓ ] Quality Education  \n[✓ ]  Industry, Innovation, and Infrastructure     [  ] Gender Equality \n[  ] Decent Work and Economic Growth           [  ]  Climate Action \nArea of specialization \n[✓ ] Artificial Intelligence (AI)      [  ] Blockchain               [  ] Cybersecurity \n[  ] Data Science and Analytics    [  ] Game Development \n[  ] Internet of Things (IoT)          [✓ ]  Natural Language Processing (NLP) \n[  ] Mobile App Development      [✓ ]  Web Development \nProject Group Members \nSr.# Reg. # Student Name Emai

### Integration VectorDB Context Pipeline with LLM Output

In [13]:
## Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [14]:
answer=rag_simple("What is the abstract of our project?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is the abstract of our project?'
Top K: 3, Max distance: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Abstract**

This project delivers an AI‑powered, data‑driven platform for higher‑education institutions that transforms static administrative systems into intelligent, adaptive learning environments. Leveraging machine‑learning models trained on student performance and feedback data, the system provides real‑time predictive analytics, trend detection, and automated report generation to support evidence‑based decision making. It offers four role‑based portals—Student, Faculty, Staff, and Admin—each equipped with personalized dashboards, adaptive content, and secure access control. Designed for rapid customization, the solution can be deployed across any university, enabling early intervention, personalized learning pathways, and continuous improvement of teaching quality and institutional effectiveness.


### Enhanced RAG Pipeline feature

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


TypeError: RAGRetriever.retrieve() got an unexpected keyword argument 'score_threshold'